In [1]:
import os

import mlflow

# то же хранилище, что и при запуске сервера
mlflow.set_tracking_uri('file:./mlflow_experiments_store')

# эксперимент по умолчанию называется Default
experiment_id = mlflow.get_experiment_by_name("Default").experiment_id

# логируем тестовую метрику и файл
with mlflow.start_run(run_name='Default', experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metric("test_metric", 0)
    mlflow.log_artifact("test_artifact.txt", "test_artifact")

print(f"Run id запуска: {run_id}")


Run id запуска: 0566fbc3c8d448339ae17d5f3a26fa7e


In [2]:
# метрики запуска — должно вывести ['test_metric']
print(os.listdir(f"mlflow_experiments_store/0/{run_id}/metrics"))

# содержимое залогированного файла — должно вывести test_artifact
print(open(f"mlflow_experiments_store/0/{run_id}/artifacts/test_artifact/test_artifact.txt").read())


['test_metric']
test_artifact



In [3]:
import sqlite3

# теперь пишем через сервер, а он сам кладёт данные в mydb.sqlite
mlflow.set_tracking_uri("http://127.0.0.1:5000")

experiment_id = mlflow.get_experiment_by_name("Default").experiment_id

with mlflow.start_run(run_name="sqlite_test", experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metric("test_metric", 0)
    mlflow.log_artifact("test_artifact.txt", "test_artifact")

print(f"Run id запуска: {run_id}")

# проверка 1: метрика лежит в базе SQLite
print(sqlite3.connect("mydb.sqlite").execute(
    "select key, value from metrics where run_uuid = ?", (run_id,)
).fetchall())

# проверка 2: файл лежит в папке артефактов
print(open(f"mlflow_experiments_store_sqlite/0/{run_id}/artifacts/test_artifact/test_artifact.txt").read())


Run id запуска: 99265fd6380245af9ab05538be246769
[('test_metric', 0.0)]
test_artifact



In [4]:
import os

import mlflow
from dotenv import load_dotenv

# читаем .env (на Linux урок делает это командой export $(cat .env | xargs))
load_dotenv()

os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://hb.ru-msk.vkcloud-storage.ru"
os.environ["AWS_ACCESS_KEY_ID"] = os.getenv("AWS_ACCESS_KEY_ID", "")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.getenv("AWS_SECRET_ACCESS_KEY", "")

TRACKING_SERVER_HOST = "127.0.0.1"
TRACKING_SERVER_PORT = 5000

YOUR_NAME = "apushkarev"  # имя для уникального эксперимента
assert YOUR_NAME, "введите своё имя в переменной YOUR_NAME"

EXPERIMENT_NAME = f"test_connection_experiment_{YOUR_NAME}"
RUN_NAME = "test_connection_run"

METRIC_NAME = "test_metric"
METRIC_VALUE = 0

mlflow.set_tracking_uri(f"http://{TRACKING_SERVER_HOST}:{TRACKING_SERVER_PORT}")

experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)
with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metric(METRIC_NAME, METRIC_VALUE)

# проверка из урока
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
run = mlflow.get_run(run_id)

assert "active" == experiment.lifecycle_stage
assert mlflow.get_run(run_id)
assert METRIC_VALUE == run.data.metrics[METRIC_NAME]

print("Все проверки пройдены, run_id:", run_id)


Все проверки пройдены, run_id: e5216ac17b9e420db3ae93d709534b84


In [6]:
import os

import mlflow
from dotenv import load_dotenv

# override=True: перечитать .env, даже если в ядре остались старые пустые ключи с прошлой ячейки
load_dotenv(override=True)
os.environ["MLFLOW_S3_ENDPOINT_URL"] = "https://hb.ru-msk.vkcloud-storage.ru"

mlflow.set_tracking_uri("http://127.0.0.1:5000")

EXPERIMENT_NAME = "churn_task_apushkarev"
RUN_NAME = "data_check"

# исправленный вариант из урока: сначала проверяем, есть ли эксперимент
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
if experiment:
    experiment_id = experiment.experiment_id
else:
    experiment_id = mlflow.create_experiment(EXPERIMENT_NAME)

# файл и папка, которые будем логировать
with open("artifacts.txt", "w") as f:
    f.write("test artifact")
os.makedirs("additional_resources", exist_ok=True)
with open("additional_resources/data_description.txt", "w") as f:
    f.write("This file contains data description")

stats = {"mae": 2, "r2": 0.82}
params = {"learning_rate": 0.01, "epochs": 100, "batch_size": 32}

with mlflow.start_run(run_name=RUN_NAME, experiment_id=experiment_id) as run:
    run_id = run.info.run_id
    mlflow.log_metrics(stats)                                  # метрики -> в Postgres
    mlflow.log_artifact("artifacts.txt", "artifacts")          # один файл -> в S3
    mlflow.log_dict(params, "model_params.json")               # словарь как JSON -> в S3
    mlflow.log_artifacts("additional_resources", "resources")  # целая папка -> в S3
    mlflow.log_text("Комментарий к эксперименту", "comment.txt")  # текст -> в S3

# проверка: что легло в базу и в S3
print("run_id:", run_id)
print("метрики из базы:", mlflow.get_run(run_id).data.metrics)

client = mlflow.MlflowClient()
def show(path=None, indent=""):
    for a in client.list_artifacts(run_id, path):
        print(indent + a.path + ("/" if a.is_dir else ""))
        if a.is_dir:
            show(a.path, indent + "  ")
print("файлы в S3:")
show()


run_id: e1297a5100494de9bffc624d9a1c9051
метрики из базы: {'mae': 2.0, 'r2': 0.82}
файлы в S3:
artifacts/
  artifacts/artifacts.txt
comment.txt
model_params.json
resources/
  resources/data_description.txt
